In [1]:

!nvidia-smi

Thu Nov 12 12:52:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("Version PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("BF16 supporté :", torch.cuda.is_bf16_supported())


Version PyTorch : 2.11.0+cu130
CUDA disponible : True
GPU : Tesla T4
BF16 supporté : True


In [3]:

!git clone -b feature/llm-client https://github.com/FatiAllali123/SLM-Software-Engineering.git

%cd /kaggle/working/SLM-Software-Engineering


fatal: destination path 'SLM-Software-Engineering' already exists and is not an empty directory.
/kaggle/working/SLM-Software-Engineering


In [4]:
!git branch

* feature/llm-client
  main


In [6]:
!git pull origin feature/llm-client

From https://github.com/FatiAllali123/SLM-Software-Engineering
 * branch            feature/llm-client -> FETCH_HEAD
Already up to date.


In [8]:
import torch

print("GPU disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("BF16 supporté :", torch.cuda.is_bf16_supported())

GPU disponible : True
GPU : Tesla T4
BF16 supporté : True


In [9]:
import torch

try:
    x = torch.randn(100, 100, device="cuda", dtype=torch.bfloat16)
    y = x @ x
    torch.cuda.synchronize()

    print("Calcul BF16 réussi !")
    print("Type :", y.dtype)

except Exception as e:
    print("Erreur BF16 :", e)

Calcul BF16 réussi !
Type : torch.bfloat16


In [ ]:
!ls llm/backends
!cat benchmarking/config.json

# Tester StubBackend

### Pour vérifier que architecture fonctionne avant d'utiliser un vrai modèle. 
 test fonctionne -----> LLMClient communique correctement avec StubBackend.

In [10]:
from llm import LLMClient
from llm.backends import StubBackend

client = LLMClient(StubBackend())

messages = [
    {"role": "user", "content": "Write a Python function to calculate factorial."}
]

result = client.generate(messages)
print(result)

{'text': '[STUB] Réponse simulée pour : Write a Python function to calculate factorial.', 'tokens_in': 7, 'tokens_out': 12, 'latency_s': 2.054000105999876e-06}


### rendre les deux backends accessibles depuis le même emplacement.

In [ ]:
%%writefile llm/backends/__init__.py

from .base import BaseBackend
from .stub import StubBackend
from .qwen_base import QwenBaseBackend

In [12]:
!cat llm/backends/__init__.py


from .base import BaseBackend
from .stub import StubBackend
from .qwen_base import QwenBaseBackend


In [ ]:

%%writefile llm/backends/qwen_base.py

import json
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from llm.backends.base import BaseBackend


class QwenBaseBackend(BaseBackend):

    def __init__(self, config_path=None):

        if config_path is None:
            project_root = Path(__file__).resolve().parents[2]
            config_path = project_root / "benchmarking" / "config.json"

        with open(config_path, "r", encoding="utf-8") as f:
            config = json.load(f)

        self.model_name = config["models"]["qwen"]
        self.generation_params = config["generation_params"]
        self.precision = config["precision"]

        if not torch.cuda.is_available():
            raise RuntimeError("Aucun GPU CUDA disponible.")

        if self.precision != "bf16":
            raise ValueError("La precision attendue est BF16.")

        if not torch.cuda.is_bf16_supported():
            raise RuntimeError("BF16 non supporte par PyTorch.")

        print("Chargement du tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )

        print("Chargement du modele Qwen en BF16...")

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.bfloat16,
            device_map={"": 0}
        )

        self.model.eval()

        print("Qwen charge avec succes.")

    def generate(self, messages: list[dict]) -> dict:

        inputs = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        )

        inputs = {
            key: value.to(self.model.device)
            for key, value in inputs.items()
        }

        tokens_in = inputs["input_ids"].shape[1]

        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.generation_params["max_new_tokens"],
                do_sample=self.generation_params["do_sample"]
            )

        torch.cuda.synchronize()
        latency_s = time.perf_counter() - start

        generated_ids = outputs[0][tokens_in:]

        response = self.tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        )

        return {
            "text": response,
            "tokens_in": int(tokens_in),
            "tokens_out": int(generated_ids.shape[0]),
            "latency_s": float(latency_s)
        }


In [16]:
!ls -l llm/backends/

total 20
-rw-r--r-- 1 root root  840 Nov 12 12:01 base.py
-rw-r--r-- 1 root root  100 Nov 12 12:49 __init__.py
drwxr-xr-x 2 root root 4096 Nov 12 12:53 __pycache__
-rw-r--r-- 1 root root 2605 Nov 12 12:47 qwen_base.py
-rw-r--r-- 1 root root 1256 Nov 12 12:01 stub.py


In [17]:
!cat llm/backends/qwen_base.py


import json
import time
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from llm.backends.base import BaseBackend


class QwenBaseBackend(BaseBackend):

    def __init__(self, config_path=None):

        if config_path is None:
            project_root = Path(__file__).resolve().parents[2]
            config_path = project_root / "benchmarking" / "config.json"

        with open(config_path, "r", encoding="utf-8") as f:
            config = json.load(f)

        self.model_name = config["models"]["qwen"]
        self.generation_params = config["generation_params"]
        self.precision = config["precision"]

        if not torch.cuda.is_available():
            raise RuntimeError("Aucun GPU CUDA disponible.")

        if self.precision != "bf16":
            raise ValueError("La precision attendue est BF16.")

        if not torch.cuda.is_bf16_supported():
            raise RuntimeError("BF16 non supporte par PyTorch.")

        pr

In [18]:
from llm.backends import QwenBaseBackend

print("Importation réussie !")

Importation réussie !


- BaseBackend : définit l'interface commune.
- StubBackend : simule un modèle.
- QwenBaseBackend : permet d'utiliser le véritable modèle Qwen.
- LLMClient : communique avec le backend choisi.
On passe maintenant au chargement du vrai modèle.

### Installer les dépendances
Pourquoi ?
- transformers : permet de charger Qwen depuis Hugging Face.
- accelerate : facilite le placement du modèle sur le GPU.

In [19]:
!pip install -q transformers accelerate

Cette instruction va :
1. Lire le fichier benchmarking/config.json.
2. Récupérer le modèle Qwen/Qwen2.5-Coder-1.5B-Instruct.
3. Charger son tokenizer.
4. Charger les poids du modèle en BF16.
5. Placer le modèle sur le GPU Kaggle.

In [20]:
backend = QwenBaseBackend()

Chargement du tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen charge avec succes.


modèle Qwen2.5-Coder-1.5B-Instruct est chargé avec succès

### Vérifier le chargement

In [21]:
print("Device :", next(backend.model.parameters()).device)
print("Précision :", next(backend.model.parameters()).dtype)

Device : cuda:0
Précision : torch.bfloat16


### veridier maintenant qu il peut reellement repondre a une question 

In [22]:
from llm import LLMClient

# Créer le client avec le vrai modèle Qwen
client = LLMClient(backend)

# Préparer le message
messages = [
    {
        "role": "user",
        "content": "Write a Python function to calculate factorial."
    }
]

# Générer la réponse
result = client.generate(messages)

# Afficher les résultats
print("Réponse :", result["text"])
print("Tokens entrée :", result["tokens_in"])
print("Tokens sortie :", result["tokens_out"])
print("Latence :", result["latency_s"], "secondes")

Réponse : ```python
def factorial(n):
    """
    Calculate the factorial of a given number n.

    Args:
    n (int): The number to calculate the factorial for.

    Returns:
    int: The factorial of the number.
    """
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n-1)
```

In this solution, the `factorial` function is defined to compute the factorial of a non-negative integer `n`. The base case of the recursion is when `n` is either 0 or 1, in which case the function returns 1. For all other values of `n`, the function calls itself with `n-1` and multiplies the result by `n`, effectively calculating the factorial. This recursive approach ensures that each call reduces the problem size until it reaches the base case.
Tokens entrée : 37
Tokens sortie : 183
Latence : 7.884483382999861 secondes


text : réponse réellement générée par Qwen.



tokens_in : nombre de tokens envoyés au modèle.



tokens_out : nombre de tokens générés.



latency_s : temps nécessaire à la génération, en secondes.

In [23]:
!ls llm/backends

base.py  __init__.py  __pycache__  qwen_base.py  stub.py


In [44]:
import os
import subprocess

env = os.environ.copy()
env["GITHUB_TOKEN"] = github_token
env["GIT_ASKPASS"] = "/tmp/git_askpass.sh"
env["GIT_TERMINAL_PROMPT"] = "0"

result = subprocess.run(
    ["git", "push", "origin", "feature/llm-client"],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Code retour :", result.returncode)


To https://github.com/FatiAllali123/SLM-Software-Engineering.git
   8ce78dd..8748049  feature/llm-client -> feature/llm-client

Code retour : 0


In [45]:
!pip install -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.8 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 113.2 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:━━━━━━━━━━━ 0/2 [transformers]
      Successfully uninstalled transformers-5.20.20m 0/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bitsandbytes] [bitsandbytes]


In [46]:
import bitsandbytes as bnb

print("BitsAndBytes installé :", bnb.__version__)

BitsAndBytes installé : 0.50.2


In [48]:
%cd /kaggle/working/SLM-Software-Engineering

/kaggle/working/SLM-Software-Engineering


In [49]:
!git branch --show-current

feature/llm-client


In [50]:
!git pull origin feature/llm-client

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 11 (delta 5), reused 6 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 7.97 KiB | 2.66 MiB/s, done.
From https://github.com/FatiAllali123/SLM-Software-Engineering
 * branch            feature/llm-client -> FETCH_HEAD
   8748049..8c966cc  feature/llm-client -> origin/feature/llm-client
Updating 8748049..8c966cc
Fast-forward
 benchmarking/notebooks/llm-backend-tests.ipynb |   1 +
 llm/backends/__init__.py                       |   1 +
 llm/backends/qwen_nf4.py                       | 115 +++++++++++++++++++++++++
 3 files changed, 117 insertions(+)
 create mode 100644 benchmarking/notebooks/llm-backend-tests.ipynb
 create mode 100644 llm/backends/qwen_nf4.py


In [51]:
from llm.backends.qwen_nf4 import QwenNF4Backend

print("Importation de QwenNF4Backend réussie !")

Importation de QwenNF4Backend réussie !


In [52]:
backend = QwenNF4Backend()

Chargement du tokenizer...
Chargement du modele Qwen en NF4...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen NF4 charge avec succes.


In [53]:
from llm.client import LLMClient

# Créer le client avec le backend NF4
client = LLMClient(backend)

# Préparer le prompt
messages = [
    {
        "role": "user",
        "content": "Explique une boucle for en Python."
    }
]

# Générer une réponse
result = client.generate(messages)

# Afficher les résultats
print("Réponse :", result["text"])
print("Tokens entrée :", result["tokens_in"])
print("Tokens sortie :", result["tokens_out"])
print("Latence :", result["latency_s"])

Réponse : Une boucle `for` en Python est utilisée pour itérer sur un ensemble de données ou sur une plage de nombres. Elle permet d'effectuer des opérations spécifiques sur chaque élément de la collection.

Voici une explication détaillée de comment fonctionne une boucle `for` en Python :

1. **Syntaxe**:
   La syntaxe générale d'une boucle `for` est :
   ```python
   for variable in iterable:
       # Code à exécuter pour chaque élément de iterable
   ```

2. **Exemple 1: Itération sur une liste**
   Voici un exemple simple qui itère sur une liste de nombres et affiche chaque nombre :
   ```python
   numbers = [1, 2, 3, 4, 5]
   for number in numbers:
       print(number)
   ```
   Dans cet exemple, `numbers` est une liste de nombres entiers. La boucle `for` parcourt chaque élément de cette liste et l'affiche dans la console.

3. **Exemple 2: Itération sur une chaîne de caractères**
   Vous pouvez également itérer sur une chaîne de caractères :
   ```python
   text = "Hello, World!"
 

les 512 tokens en sortie correspondent exactement à max_new_tokens=512 dans la configuration. Cela indique que le modèle a probablement atteint la limite de génération avant de terminer sa réponse. Ce n'est pas une erreur du backend.

In [54]:
print("Modèle chargé en 4 bits :", backend.model.is_loaded_in_4bit)

print("Configuration de quantification :")
print(backend.model.config.quantization_config)

Modèle chargé en 4 bits : True
Configuration de quantification :
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [55]:
memory_mb = torch.cuda.memory_allocated() / (1024 ** 2)

print(f"Mémoire GPU utilisée : {memory_mb:.2f} MB")

Mémoire GPU utilisée : 1109.97 MB


Paramètre	Valeur	Signification
load_in_4bit	True	Modèle chargé en 4 bits
bnb_4bit_quant_type	nf4	Quantification utilisée : NF4
bnb_4bit_use_double_quant	True	Double quantification activée
bnb_4bit_compute_dtype	bfloat16	Calculs effectués en BF16
bnb_4bit_quant_storage	uint8	Stockage des valeurs quantifiées

À retenir :
- NF4 réduit la précision de stockage des poids à 4 bits.
- BF16 est conservé pour les calculs.
- Double quantification permet de réduire davantage la mémoire nécessaire.